<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-6-evaluation-monitoring/Module_6_Session_2_Opik_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6 — Session 2: Opik Monitoring

## What we're building
LangSmith tracks speed and tokens. Opik tracks *quality* —
was the answer relevant? did it hallucinate? was the tone right?
Together they give full observability: performance + quality.

## The analogy
LangSmith = speedometer and fuel gauge (how fast, how much)
Opik = dashcam with AI analysis (what actually happened, was it good?)

## AWS equivalent
Amazon CloudWatch (metrics) + AWS X-Ray (traces) +
Amazon Bedrock Model Evaluation (quality scoring)

## Step 0: Install Opik
Opik is built by Comet ML. The library handles both logging traces
and scoring LLM responses for quality metrics.
One new library this session: opik

In [ ]:
# Install opik — the LLM quality monitoring library
# -q means quiet (less output spam)
!pip install -q opik
print("✅ Opik installed")

## Step 1: Configure Opik
We load our API key and tell Opik which workspace and project to log to.
Workspace = your Comet ML account's workspace (usually your username)
Project = the folder where traces get grouped (like LangSmith's project name)

In [ ]:
import os
from google.colab import userdata
import opik

os.environ["OPIK_API_KEY"] = userdata.get("OPIK_API_KEY")

# force=True skips the interactive prompt — no Y/n questions
opik.configure(
    api_key=userdata.get("OPIK_API_KEY"),
    workspace="shaif-tazir",
    use_local=False,
    force=True                          # ← skips interactive prompts
)

print("✅ Opik configured")

In [ ]:
!pip install -q langchain-groq opik
print("✅ Libraries installed")

## Step 2: Log a traced call to Opik
Opik has a built-in LangChain integration — one line sets it up.
OpikTracer() is a callback that LangChain passes to the LLM call.
Every call with this callback gets automatically logged to Opik.

In [ ]:
from langchain_groq import ChatGroq
from opik.integrations.langchain import OpikTracer

# Load Groq key
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# OpikTracer is a LangChain callback — it intercepts every LLM call
# and logs it to Opik automatically
# project_name overrides whatever the config file says
tracer = OpikTracer(project_name="swiggy-module-6")

# Set up the LLM — same as always
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

# Make the call — notice: callbacks=[tracer] is the only new thing
# This tells LangChain to pass every call through OpikTracer
response = llm.invoke(
    "My Swiggy order arrived cold and 45 mins late. I want a refund.",
    config={"callbacks": [tracer]}      # ← this is how Opik hooks in
)

print("Reply:", response.content[:100], "...")

# Flush — same concept as LangSmith, push pending traces now
tracer.flush()
print("✅ Trace sent to Opik")

## Step 3: LLM Quality Scoring with Opik
LangSmith tracks latency and tokens.
Opik adds *quality scores* — automated checks that judge whether
the LLM response was actually good.

We'll score three things on every Swiggy support reply:
1. Hallucination — did the model make up any facts?
2. Answer relevance — did it actually address the complaint?
3. Moderation — is the response appropriate and safe?

These are Opik's built-in scoring metrics — no prompt engineering needed.

In [ ]:
from opik.evaluation.metrics import Hallucination, AnswerRelevance, Moderation

# These are Opik's built-in LLM-as-Judge scorers
# Each one sends a scoring prompt to an LLM internally and returns a 0-1 score
hallucination_scorer = Hallucination()     # did the model make up facts?
relevance_scorer = AnswerRelevance()       # did it answer what was asked?
moderation_scorer = Moderation()           # is the content appropriate?

print("✅ Scorers loaded")
print("Hallucination scorer:", hallucination_scorer.name)
print("Relevance scorer:", relevance_scorer.name)
print("Moderation scorer:", moderation_scorer.name)

## Step 4: Score a Swiggy support reply
We run our three scorers on the actual complaint and reply.
Each scorer uses LLM-as-Judge internally — it sends a scoring
prompt to an LLM and gets back a 0-1 score with a reason.
This is automated quality checking at scale.

In [ ]:
from opik.evaluation.metrics import Hallucination, AnswerRelevance, Moderation
import os
from google.colab import userdata

# Opik scorers use LiteLLM internally
# LiteLLM supports Groq — we just need to tell it which model and pass the key
# LiteLLM's Groq model string format is: "groq/model-name"
groq_model = "groq/llama-3.3-70b-versatile"   # a Groq model LiteLLM knows about

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Pass the model name to each scorer so they use Groq, not OpenAI
hallucination_scorer = Hallucination(model=groq_model)
relevance_scorer = AnswerRelevance(model=groq_model)
moderation_scorer = Moderation(model=groq_model)

print("✅ Scorers loaded with Groq backend")

In [ ]:
# Fix: require_context=False tells the scorer to judge relevance
# based only on the input question and output answer
# no RAG context needed
relevance_scorer = AnswerRelevance(
    model=groq_model,
    require_context=False               # ← this is the fix
)

# Now score all three — hallucination already worked, re-running for completeness
complaint = "My Swiggy order arrived cold and 45 mins late. I want a refund."
reply = """I'm sorry to hear that your order didn't meet your expectations.
To help you with a refund, could you please share your order ID?
Once I have that, I'll process the refund for you."""

# Hallucination
h = hallucination_scorer.score(input=complaint, output=reply)
print("=== Hallucination ===")
print("Score:", h.value, "| Reason:", h.reason[0] if isinstance(h.reason, list) else h.reason)

# Relevance
r = relevance_scorer.score(input=complaint, output=reply)
print("\n=== Answer Relevance ===")
print("Score:", r.value, "| Reason:", r.reason)

# Moderation
m = moderation_scorer.score(input=complaint, output=reply)
print("\n=== Moderation ===")
print("Score:", m.value, "| Reason:", m.reason)

print("\n✅ All three scores complete")

## Step 5: Log quality scores to Opik dashboard
Scoring locally is useful. But logging scores to Opik means you can
track quality trends over time — did our prompt changes improve relevance?
Did hallucination increase after a model update?
This is production-grade LLM quality monitoring.

In [ ]:
import opik

# Get the Opik client to log feedback scores
opik_client = opik.Opik()

# First make a traced call so we have a trace to attach scores to
tracer = OpikTracer(project_name="swiggy-module-6")

llm_response = llm.invoke(
    complaint,
    config={"callbacks": [tracer]}
)
tracer.flush()

# Get the trace ID of the call we just made
trace_id = tracer.created_traces()[0].id
print("Trace ID:", trace_id)

# Log all three scores to that trace on the Opik dashboard
opik_client.log_traces_feedback_scores(
    scores=[
        {"id": trace_id, "name": "hallucination",     "value": h.value},
        {"id": trace_id, "name": "answer_relevance",  "value": r.value},
        {"id": trace_id, "name": "moderation",        "value": m.value},
    ]
)

print("✅ Scores logged to Opik dashboard")
print(f"  hallucination:    {h.value}")
print(f"  answer_relevance: {r.value}")
print(f"  moderation:       {m.value}")

In [ ]:
# Check what trace ID we actually logged scores to
print("We logged scores to trace:", trace_id)

# Check what traces the tracer created
all_traces = tracer.created_traces()
print("Traces created by tracer:")
for t in all_traces:
    print(" - ID:", t.id)

## Session Summary

### What we built
A complete Opik monitoring setup for a Swiggy support pipeline:
1. Connected Opik to our Colab notebook (APAC-friendly, no region issues)
2. Logged LLM traces using OpikTracer — LangChain callback integration
3. Scored replies using three built-in LLM-as-Judge metrics:
   - Hallucination: 0.0 (no hallucination — reply made no factual claims)
   - Answer Relevance: 0.9 (highly relevant — docked 0.1 for asking order ID)
   - Moderation: 0.0 (clean — no content policy violations)
4. Logged scores back to Opik dashboard via SDK

### Key concepts
- OpikTracer: LangChain callback that auto-logs every LLM call to Opik
- LLM-as-Judge: scorers use an LLM internally to evaluate another LLM's output
- Feedback scores: quality metrics attached to traces for trend monitoring
- require_context=False: needed when scoring without RAG retrieved context

### LangSmith vs Opik — when to use which
LangSmith: latency, tokens, cost, debugging step-by-step chains
Opik: hallucination, relevance, moderation — quality of LLM responses
Production systems use BOTH together for full observability

### Groq + LiteLLM fix
Opik scorers use LiteLLM internally and default to OpenAI.
Fix: pass model="groq/llama-3.3-70b-versatile" to each scorer.

### AWS equivalent
LangSmith → AWS X-Ray + CloudWatch
Opik → Amazon Bedrock Model Evaluation + CloudWatch custom metrics

### What is next
Module 6 Session 3 — Golden Set Evaluation:
Building a curated test dataset and running systematic evaluation
across multiple prompts — not just monitoring live traffic,
but testing before deployment.